In [ ]:
from NetworkExpansion import *

results = dict()
reaction_scopes = {'original': kegg_2018_0, 'original_noflagged': remove_questionable(kegg_2018_0), 
                   'updated': kegg_2024_0, 'updated_noflagged': remove_questionable(kegg_2024_0)}
for name, reaction_scope in reaction_scopes.items():
    results[name] = expand_from_seed_set(seeds=primordial_seed_set, reactions=reaction_scope, max_gen=80)

def result_to_df(key:str, to_apply):
    return pd.concat({i: results[i][key].apply(to_apply) for i in results.keys()}, axis=1)
summaries = {'compound count': result_to_df(key='cps', to_apply=lambda x: x.sum()),
            'reaction count': result_to_df(key='rns', to_apply=lambda x: x.sum()),
            'chiral fraction': result_to_df(key='chiral_fraction', to_apply=lambda x: x)}

for name, reaction_scope in reaction_scopes.items():
    network = nx.from_pandas_edgelist(generate_edge_df(reaction_scope), create_using=nx.Graph, source='left', target='right', edge_attr=None)
    reachable_cps = set.union(*[nx.node_connected_component(network, i) for i in results[name]['seeds']]) #everything reachable from at least one seed
    results[name]['reachable'] = network.subgraph(reachable_cps)
reachable = pd.Series({name: len(results[name]['reachable']) for name in results.keys()})
summaries['reached/reachable'] = summaries['compound count'].div(reachable, axis=1)

fig, ax = plt.subplots(len(summaries.keys()),1, figsize=(6,10), sharex=True, layout='compressed')
for ict, iname in enumerate(summaries.keys()):
    summaries[iname].plot(ax=ax[ict], ylabel=iname, cmap='coolwarm').legend(loc='lower right')
plt.show()

edges = {i: generate_edge_df(reaction_scopes[i].loc[results[i]['rns'].index]) for i in reaction_scopes}
networks = {i: nx.from_pandas_edgelist(edges[i], create_using=nx.Graph, source='left', target='right', edge_attr=None) for i in edges.keys()}
degrees = pd.DataFrame({i: pd.Series(dict(networks[i].degree())) for i in networks.keys()})